<a href="https://colab.research.google.com/github/Mru321/CSI-Cross-Environment-Generalization/blob/main/Random%20Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
dataset_path = "/content/drive/MyDrive/Project/Project1/processed/csi_feature_dataset.csv"

df = pd.read_csv(dataset_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (53016, 9)


,mean_amp,std_amp,var_amp,max_amp,min_amp,antenna_corr,subcarrier_corr,label,environment
0,2.488678,0.579163,0.335430,3.975699,0.559248,0.542688,0.815294,1,processed_channe14_LOS_cluster1.hdf5
1,2.488807,0.575728,0.331463,3.883031,0.569000,0.550662,0.816676,1,processed_channe14_LOS_cluster1.hdf5
2,2.489268,0.579336,0.335631,3.982258,0.566464,0.538711,0.818489,1,processed_channe14_LOS_cluster1.hdf5
3,2.489166,0.579777,0.336141,4.024301,0.540201,0.526773,0.818690,1,processed_channe14_LOS_cluster1.hdf5
4,2.489647,0.577595,0.333616,3.971815,0.537490,0.534651,0.819378,1,processed_channe14_LOS_cluster1.hdf5


In [ ]:
train_env = [
    "processed_channe14_LOS_cluster1.hdf5",
    "processed_channe14_LOS_cluster2.hdf5",
    "processed_channe14_LOS_cluster3.hdf5",
    "processed_channe14_NLOS_cluster1.hdf5",
    "processed_channe14_NLOS_cluster2.hdf5",
    "processed_channe14_NLOS_cluster3.hdf5"
]

test_env = [
    "processed_channe14_LOS_cluster4.hdf5",
    "processed_channe14_NLOS_cluster4.hdf5",
    "processed_channe14_NLOS_cluster5.hdf5"
]

In [ ]:
train_df = df[df["environment"].isin(train_env)]
test_df  = df[df["environment"].isin(test_env)]

print("Train samples:", train_df.shape)
print("Test samples:", test_df.shape)

Train samples: (37398, 9)
Test samples: (15618, 9)


In [ ]:
X_train = train_df.drop(columns=["label", "environment"])
y_train = train_df["label"]

X_test = test_df.drop(columns=["label", "environment"])
y_test = test_df["label"]

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
rf_model = RandomForestClassifier(
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

**Doing Accuracy check on each test environment individually**

In [ ]:
env_accuracy = []

for env in test_env:

    env_data = df[df["environment"] == env]

    X_env = env_data.drop(columns=["label", "environment"])
    y_env = env_data["label"]

    X_env_scaled = scaler.transform(X_env)

    y_pred = rf_model.predict(X_env_scaled)

    acc = accuracy_score(y_env, y_pred)

    env_accuracy.append((env, acc))

In [ ]:
env_df = pd.DataFrame(env_accuracy, columns=["Environment", "Accuracy"])

env_df = env_df.sort_values(by="Accuracy")

print(env_df)

                             Environment  Accuracy
1  processed_channe14_NLOS_cluster4.hdf5  0.996110
2  processed_channe14_NLOS_cluster5.hdf5  0.999741
0   processed_channe14_LOS_cluster4.hdf5  1.000000
